In [0]:
#################################################
##############cash_flow##########################
#################################################
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, expr, current_timestamp,
    to_date, from_unixtime, year, regexp_extract, first
)
from pyspark.sql.types import StructType
from functools import reduce
from pyspark.sql import DataFrame

spark = SparkSession.builder.getOrCreate()

# key metrics for cashflow
key_metrics = [
    "Operating_Cash_Flow",
    "Capital_Expenditure",
    "Free_Cash_Flow",
    "Cash_Dividends_Paid",
    "Depreciation_And_Amortization",
    "Depreciation",
    "Net_Income_From_Continuing_Operations",
    "Repurchase_Of_Capital_Stock",
    "Repayment_Of_Debt",
    "Issuance_Of_Debt",
    "Beginning_Cash_Position",
    "End_Cash_Position",
    "Changes_In_Cash",
    "Stock_Based_Compensation",
    "Change_In_Working_Capital",
    "Purchase_Of_PPE",
    "Sale_Of_PPE",
    "Purchase_Of_Business",
    "Sale_Of_Business",
    "Common_Stock_Payments"
]


df = spark.read.table("investment_intelligence_platform.bronze.cashflow")


df = df.withColumn(
    "ticker",
    regexp_extract(col("meta_data"), r'([^/]+)\.json$', 1)
)

# identify struct columns 
metric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, StructType)
    and f.name in key_metrics
]

#   melt wide to long 
rows = []
for metric in metric_cols:
    metric_date_cols = [f.name for f in df.schema[metric].dataType.fields]
    stack_expr = f"stack({len(metric_date_cols)}, " + \
        ", ".join([f"'{d}', CAST({metric}.`{d}` AS DOUBLE)"
                   for d in metric_date_cols]) + ")"
    row = df.select(
        col("ticker"),
        lit(metric).alias("metric"),
        col("last_updated_ts"),
        expr(stack_expr).alias("fiscal_date_unix", "amount")
    )
    rows.append(row)

melted = reduce(DataFrame.union, rows)

#   convert dates, clean 
long_df = melted \
    .withColumn("fiscal_date",
        to_date(from_unixtime(col("fiscal_date_unix").cast("long") / 1000))
    ) \
    .withColumn("fiscal_year", year(col("fiscal_date"))) \
    .filter(col("amount").isNotNull()) \
    .drop("fiscal_date_unix")

#  pivot metrics as columns 
silver_df = long_df \
    .groupBy("ticker", "fiscal_date", "fiscal_year", "last_updated_ts") \
    .pivot("metric", key_metrics) \
    .agg(first("amount")) \
    .withColumn("silver_updated_ts", current_timestamp()) \
    .orderBy("ticker", "fiscal_date")


# save to silver 
spark.sql("CREATE SCHEMA IF NOT EXISTS investment_intelligence_platform.silver")

from delta.tables import DeltaTable

def scd_merge_table(spark, source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        print(f"First load — creating table: {target_table}")
        source_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)
        print("Table created ")
    else:
        print(f"Incremental load — merging: {target_table}")
        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )
        delta_table = DeltaTable.forName(spark, target_table)
        delta_table.alias("target") \
            .merge(source_df.alias("source"), merge_condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print("Merge completed ")

scd_merge_table(
    spark,
    silver_df,
    "investment_intelligence_platform.silver.cashflow",
    ["ticker", "fiscal_date"]
)
print(" Silver cashflow saved successfully")